In [44]:
#importamos librerias y dependencias y las configuraciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os


#CARGAMOS EL DATASET
df = pd.read_csv('/content/data_v02_clustering_limpio_ramon_v2.csv')
display(df.sample(5))

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
62311,544811,22720,SET OF 3 CAKE TINS PANTRY DESIGN,6,2011-02-23,4.95,12471,Germany,29.70
162767,557743,84978,HANGING HEART JAR T-LIGHT HOLDER,12,2011-06-22,1.25,18155,United Kingdom,15.00
310977,573772,21559,STRAWBERRY LUNCH BOX WITH CUTLERY,1,2011-11-01,2.55,17377,United Kingdom,2.55
219582,564553,22429,ENAMEL MEASURING JUG CREAM,2,2011-08-25,4.25,18200,United Kingdom,8.50
118362,551992,85062,PEARL CRYSTAL PUMPKIN T-LIGHT HLDR,12,2011-05-05,1.65,12747,United Kingdom,19.80


In [45]:
print(f"Filas originales antes de limpieza: {len(df)}")

# FASE 1: LIMPIEZA DE DATOS

# Eliminar CustomerID nulos
df_clean = df.dropna(subset=['CustomerID']).copy()

# Eliminar Descripciones nulas
df_clean = df_clean.dropna(subset=['Description'])

# Eliminar Invoices Canceladas (Nos quedamos solo con las ventas)
# Asumimos que las cancelaciones tienen Quantity negativa o empiezan por 'C' en el InvoiceNo
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Convertir CustomerID a string
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int).astype(str)

# Convertir la fecha
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Calcular el TotalPrice de cada línea
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Eliminar las variables de regresión
columnas_regresion = ['Sales_Lag_1', 'Sales_Lag_7', 'Sales_Mean_7', 'Sales_Mean_30', 'dayOfweek', 'is_weekend', 'DailyTotal', 'Trend_Lag1_vs_Lag7', 'IsUk', 'DayOfWeek','IsWeekend']
columnas_a_borrar = [col for col in columnas_regresion if col in df_clean.columns]

if columnas_a_borrar:
    df_clean = df_clean.drop(columns=columnas_a_borrar)
    print(f"Columnas eliminadas: {columnas_a_borrar}")

print(f"Filas resultantes tras limpieza: {len(df_clean)}")
display(df_clean.head())

Filas originales antes de limpieza: 390600
Filas resultantes tras limpieza: 390600


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01,2.55,17850,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01,3.39,17850,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01,2.75,17850,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01,3.39,17850,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01,3.39,17850,United Kingdom,20.34


In [46]:
# FASE 2: FEATURE ENGINEERING

# 1. Preparación de variables temporales en el dataset limpio
fecha_referencia = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)
df_clean['DayOfWeek'] = df_clean['InvoiceDate'].dt.dayofweek
df_clean['Is_Weekend'] = np.where(df_clean['DayOfWeek'] >= 5, 1, 0)

# 2. Agrupación y creación de variables Core
df_usuarios = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': ['max', 'min'],
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum',
    'StockCode': 'nunique',
    'Quantity': 'sum',
    'Is_Weekend': 'mean',
    'Country': 'first'
    # Eliminado el agregado de DayOfWeek (Favorite_Day) — se descarta para HDBSCAN
}).reset_index()

# Aplanamos los nombres de las columnas
df_usuarios.columns = ['CustomerID', 'LastPurchase', 'FirstPurchase', 'Frequency',
                       'Monetary', 'UniqueProducts', 'TotalQuantity',
                       'Weekend_Ratio', 'Country']

# 3. Creación de variables calculadas

# RFM + Tenure
df_usuarios['Recency'] = (fecha_referencia - df_usuarios['LastPurchase']).dt.days
df_usuarios['Tenure'] = (fecha_referencia - df_usuarios['FirstPurchase']).dt.days

# Frecuencia avanzada y Ticket
df_usuarios['AverageOrderValue'] = df_usuarios['Monetary'] / df_usuarios['Frequency']
df_usuarios['Products_Per_Order'] = df_usuarios['UniqueProducts'] / df_usuarios['Frequency']
# Eliminado AvgQuantityPerOrder (correlacionado con AverageOrderValue)

# Ciclo de compra
df_usuarios['Avg_Days_Between_Purchases'] = np.where(df_usuarios['Frequency'] > 1,
                                                     df_usuarios['Tenure'] / df_usuarios['Frequency'], 0)

# Variables categóricas/binarias
df_usuarios['Is_UK'] = np.where(df_usuarios['Country'] == 'United Kingdom', 1, 0)
df_usuarios['Weekend_Shopper'] = np.where(df_usuarios['Weekend_Ratio'] > 0.5, 1, 0)

# Media mensual de compras
meses_activos = df_clean.groupby('CustomerID')['InvoiceDate'].apply(
    lambda x: x.dt.to_period('M').nunique()
).reset_index(name='ActiveMonths')
df_usuarios = pd.merge(df_usuarios, meses_activos, on='CustomerID', how='left')
df_usuarios['Monthly_Average_Spend'] = np.where(df_usuarios['ActiveMonths'] > 0,
                                                df_usuarios['Monetary'] / df_usuarios['ActiveMonths'], 0)

# 4. Limpieza final de columnas redundantes
# Eliminamos:
# - LastPurchase, FirstPurchase: ya resumidas en Recency y Tenure
# - TotalQuantity: correlación alta con Monetary
# - Tenure: correlación alta con Frequency (después de calcular variables derivadas)
# - Weekend_Ratio: ya tenemos Weekend_Shopper como binaria
# - Country: ya tenemos Is_UK
# - ActiveMonths: ya resumida en Monthly_Average_Spend
columnas_a_borrar = ['LastPurchase', 'FirstPurchase', 'TotalQuantity', 'Tenure',
                     'Weekend_Ratio', 'Country', 'ActiveMonths']
df_usuarios = df_usuarios.drop(columns=columnas_a_borrar)

# Asegurar que no hay infinitos ni nulos
df_usuarios = df_usuarios.replace([np.inf, -np.inf], 0).fillna(0)

print(f"Número de clientes listos para segmentar: {len(df_usuarios)}")
print(f"Número de variables predictivas: {len(df_usuarios.columns) - 1}")
display(df_usuarios.head())

Número de clientes listos para segmentar: 4333
Número de variables predictivas: 10


,CustomerID,Frequency,Monetary,UniqueProducts,Recency,AverageOrderValue,Products_Per_Order,Avg_Days_Between_Purchases,Is_UK,Weekend_Shopper,Monthly_Average_Spend
0,12347,7,4310.00,103,3,615.714286,14.714286,52.571429,0,0,615.714286
1,12348,4,1437.24,21,76,359.310000,5.250000,89.750000,0,0,359.310000
2,12349,1,1457.55,72,19,1457.550000,72.000000,0.000000,0,0,1457.550000
3,12350,1,294.40,16,311,294.400000,16.000000,0.000000,0,0,294.400000
4,12352,7,1385.74,57,37,197.962857,8.142857,42.428571,0,0,346.435000


In [47]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

# FASE 3: OUTLIERS + ESCALADO
# Eliminado el One-Hot Encoding (Favorite_Day ya no existe)
# Esto reduce 7 columnas extra y mejora HDBSCAN al bajar dimensionalidad

# Separar el CustomerID para que el algoritmo no lo use
clientes_id = df_usuarios['CustomerID']
datos_modelo = df_usuarios.drop(columns=['CustomerID'])

# DETECCIÓN DE OUTLIERS (Isolation Forest)
print("Buscando anomalías con Isolation Forest...")
iso_forest = IsolationForest(contamination=0.05, random_state=42)
outlier_labels = iso_forest.fit_predict(datos_modelo)

mask_normales = (outlier_labels == 1)
datos_limpios = datos_modelo[mask_normales]
clientes_id_limpios = clientes_id[mask_normales].reset_index(drop=True)

print(f"-> Se han eliminado {len(datos_modelo) - len(datos_limpios)} clientes atípicos (outliers).")

# NORMALIZACIÓN (Robust Scaling)
print("Escalando variables con RobustScaler...")
scaler = RobustScaler()
datos_escalados = scaler.fit_transform(datos_limpios)

df_escalado = pd.DataFrame(datos_escalados, columns=datos_limpios.columns)
df_modelo_final = pd.concat([clientes_id_limpios, df_escalado], axis=1)

print(f"-> ¡Datos listos y escalados con éxito!")
print(f"Dimensiones finales: {df_escalado.shape}")
display(df_modelo_final.head())

Buscando anomalías con Isolation Forest...
-> Se han eliminado 217 clientes atípicos (outliers).
Escalando variables con RobustScaler...
-> ¡Datos listos y escalados con éxito!
Dimensiones finales: (4116, 10)


,CustomerID,Frequency,Monetary,UniqueProducts,Recency,AverageOrderValue,Products_Per_Order,Avg_Days_Between_Purchases,Is_UK,Weekend_Shopper,Monthly_Average_Spend
0,12347,1.666667,3.169621,1.189655,-0.408333,1.483324,0.046902,0.271825,-1.0,0.0,1.077636
1,12348,0.666667,0.699720,-0.224138,0.200000,0.361679,-0.574555,0.788194,-1.0,0.0,0.170240
2,12350,-0.333333,-0.282854,-0.310345,2.158333,0.077730,0.131327,-0.458333,-1.0,0.0,-0.059472
3,12352,1.666667,0.655442,0.396552,-0.125000,-0.344136,-0.384600,0.130952,-1.0,0.0,0.124676
4,12353,-0.333333,-0.459450,-0.517241,1.275000,-0.820796,-0.656635,-0.458333,-1.0,0.0,-0.786367


In [ ]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import plotly.express as px


# FASE 4: ENTRENAMIENTO Y VISUALIZACIÓN DE MODELO HDBSCAN
# Grid Search 2D: min_samples x min_cluster_size

min_samples_values = np.logspace(np.log10(2), np.log10(80), 15, dtype=int)
min_cluster_sizes = np.logspace(np.log10(2), np.log10(80), 15, dtype=int)

# Eliminar duplicados que genera logspace con enteros
min_samples_values = sorted(set(min_samples_values))
min_cluster_sizes = sorted(set(min_cluster_sizes))

resultados = []
heatmap_data = pd.DataFrame(index=min_samples_values, columns=min_cluster_sizes, dtype=float)

best_score = -1
best_params = {}
best_labels = None

for mcs in min_cluster_sizes:
    for ms in min_samples_values:
        model = HDBSCAN(min_samples=ms, min_cluster_size=mcs)
        labels = model.fit_predict(df_escalado)

        mask = labels != -1
        n_clusters = len(set(labels[mask]))
        n_ruido = sum(labels == -1)
        pct_ruido = round(n_ruido / len(labels) * 100, 1)

        if n_clusters < 2:
            score = -1
        else:
            score = silhouette_score(df_escalado[mask], labels[mask])

        heatmap_data.loc[ms, mcs] = score if score > -1 else np.nan

        resultados.append({
            'min_samples': ms,
            'min_cluster_size': mcs,
            'silhouette': round(score, 4) if score > -1 else None,
            'n_clusters': n_clusters,
            'n_ruido': n_ruido,
            'pct_ruido': pct_ruido
        })

        if score > best_score:
            best_score = score
            best_params = {'min_samples': ms, 'min_cluster_size': mcs, 'score': score}
            best_labels = labels.copy()

# DataFrame de resultados
df_resultados = pd.DataFrame(resultados)

# Heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(heatmap_data.astype(float), annot=True, fmt=".2f", cmap="YlGnBu",
            linewidths=0.5, cbar_kws={'label': 'Silhouette Score'})
plt.title('HDBSCAN - Grid Search: Silhouette Score por combinación de hiperparámetros')
plt.xlabel('min_cluster_size')
plt.ylabel('min_samples')
plt.tight_layout()
plt.show()

# Resultados del mejor modelo
best_mask = best_labels != -1
print(f"Mejor combinación:")
print(f"  min_samples:      {best_params['min_samples']}")
print(f"  min_cluster_size: {best_params['min_cluster_size']}")
print(f"  Silhouette Score: {best_params['score']:.4f}")
print(f"  Clusters:         {len(set(best_labels[best_mask]))}")
print(f"  Puntos de ruido:  {sum(best_labels == -1)} de {len(best_labels)} ({round(sum(best_labels == -1)/len(best_labels)*100,1)}%)")

# Tabla de todos los resultados ordenada
display(df_resultados.sort_values('silhouette', ascending=False).head(20))


# Visualización 2D con PCA
pca_2d_hdbscan = PCA(n_components=2, random_state=42)
X_pca_2d_hdbscan = pca_2d_hdbscan.fit_transform(df_escalado)

df_pca_2d = pd.DataFrame({
    "CustomerID": clientes_id_limpios.values,
    "PC1": X_pca_2d_hdbscan[:, 0],
    "PC2": X_pca_2d_hdbscan[:, 1],
    "Cluster": best_labels.astype(str)
})

plt.figure(figsize=(9, 6))
sns.scatterplot(
    data=df_pca_2d,
    x="PC1",
    y="PC2",
    hue="Cluster",
    alpha=0.7,
    s=35
)
plt.title("HDBSCAN - Visualización PCA 2D")
plt.xlabel(f"PC1 ({pca_2d_hdbscan.explained_variance_ratio_[0]:.2%})")
plt.ylabel(f"PC2 ({pca_2d_hdbscan.explained_variance_ratio_[1]:.2%})")
plt.legend(title="Cluster HDBSCAN", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

print(f"Varianza explicada PC1+PC2: {sum(pca_2d_hdbscan.explained_variance_ratio_)*100:.1f}%")

